# 06(EQL) — BagZIT-EQL seed sweep (source best params + iso/tail calibration)

`bag-zit-eql-final-{SRC_NUM}` Optuna study(BagZIT-EQL, **Tweedie unit deviance** phi 타깃)의 best trial HP를 그대로 받아 **여러 seed로 5-fold OOF를 다시 학습**하고, unit 후처리 + isotonic/tail 보정 후보를 훑어 seed별 val/test RMSE를 기록한다. (`06_bag_zit_seed_sweep`의 EQL 변종 — `02_bag_zit_eql_parallel_hpo.py`의 `BagZITEQLRegressor`를 재현.)

- **소스 전환**: `## 1` 셀의 `SRC_NUM`/`SRC_VARIANT`로 study명·db경로·모델클래스가 모두 바뀐다. 기본은 `SRC_NUM='002'`, `SRC_VARIANT='eql'`.
- **★ base와의 차이 (중요)**: phi M-step 타깃이 base BagZIT의 Pearson `(y-mu)²/mu^zeta`가 아니라 **Tweedie unit deviance**. `SRC_VARIANT='eql'`이면 워커의 `BagZITEQLRegressor`를 로드해 재학습하므로 best params를 정확히 재현한다 (base `BagZITboostRegressor`로는 재현 불가).
- **BagZIT 공통**: ① `fit(X, y, unit_id)` — unit_id 필수, ② die→unit 집계는 **SUM** (각 die가 unit_y/n_die 몫을 학습하므로 합쳐야 unit 예측), ③ tau_pi는 die-level structural-zero gate.
- **전처리**: source의 `pp_fixed` 재사용. EQL 병렬워커의 median-fallback impute는 `USE_MEDIAN_IMPUTE_PATCH='auto'`가 study 메타로 자동 판별.
- **출력**: `4_output/01_zit/bag_zit/seed_sweep_{SRC_NUM}/<run>/` (seed_sweep_summary.csv + best/ 산출물 9종)

## 0. 환경 설정


In [8]:
from pathlib import Path
import gc
import hashlib
import itertools
import json
import os
import pickle
import runpy
import sys
import time
from datetime import datetime

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand / 'setup.py').exists() and (cand / 'utils').exists():
            return cand
    raise RuntimeError('프로젝트 루트를 찾지 못함: setup.py + utils/ 기준')

ROOT = find_project_root()
runpy.run_path(str(ROOT / 'setup.py'))

from utils.config import (  # setup.py 실행 뒤 import하므로 noqa 유지
    PROJECT_ROOT as CFG_PROJECT_ROOT,
    OUTPUT_DIR,
    TARGET_COL,
    KEY_COL,
    DIE_KEY_COL,
    SEED as DEFAULT_SEED,
)
from utils.data import load_all, get_feat_cols, split_xs  # setup.py 실행 뒤 import하므로 noqa 유지

PP_DIR = Path(CFG_PROJECT_ROOT) / '2_preprocessing'
if str(PP_DIR) not in sys.path:
    sys.path.insert(0, str(PP_DIR))

MOD_DIR = Path(CFG_PROJECT_ROOT) / '3_modeling'
if str(MOD_DIR) not in sys.path:
    sys.path.insert(0, str(MOD_DIR))

from meta_features import add_meta_features  # setup.py 실행 뒤 import하므로 noqa 유지
from modules import preprocess, postprocess  # setup.py 실행 뒤 import하므로 noqa 유지
from modules.zit import BagZITboostRegressor  # setup.py 실행 뒤 import하므로 noqa 유지
from sklearn.isotonic import IsotonicRegression  # setup.py 실행 뒤 import하므로 noqa 유지
from sklearn.model_selection import KFold  # setup.py 실행 뒤 import하므로 noqa 유지
from scipy.interpolate import PchipInterpolator  # PCHIP smoothing용

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

PROJECT_ROOT = Path(CFG_PROJECT_ROOT)
OUTPUT_DIR = Path(OUTPUT_DIR)
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'OUTPUT_DIR   = {OUTPUT_DIR}')

setup 완료
PROJECT_ROOT = C:\Users\Dell5371\Desktop\기업연계프로젝트
OUTPUT_DIR   = C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output


## 1. 실행 설정


In [9]:
# 소스: BagZIT(-EQL) Optuna study(bag-zit-{SRC_VARIANT-}final-{SRC_NUM})의 best trial params를 db에서 직접 로드한다.
# ★ SRC_NUM/SRC_VARIANT만 바꾸면 study명/db경로/모델클래스/산출물폴더가 전부 따라 바뀐다.
#   - SRC_VARIANT='eql' : bag-zit-eql-final-* (02_bag_zit_eql_parallel_hpo.py, BagZITEQLRegressor, median-fallback impute)
#   - SRC_VARIANT=''     : bag-zit-final-*     (02_bag_zit[_parallel_hpo], base BagZITboostRegressor)
SRC_NUM = '002'
SRC_VARIANT = 'eql'   # '' = base BagZIT(Pearson phi) / 'eql' = BagZIT-EQL(Tweedie unit deviance phi)

_variant_infix = f'{SRC_VARIANT}-' if SRC_VARIANT else ''
SOURCE_STUDY_NAME = f'bag-zit-{_variant_infix}final-{SRC_NUM}'
SOURCE_DB_DIR = Path(OUTPUT_DIR) / '01_zit' / 'bag_zit' / 'hp' / SRC_NUM
SOURCE_DB_PATH = SOURCE_DB_DIR / f'optuna_jh_{SOURCE_STUDY_NAME}.db'
# save_result_artifacts의 provenance(source_best_dir)에 소스 db 경로를 기록한다.
SOURCE_BEST_DIR = SOURCE_DB_PATH

MODEL_NAME = f'bag_zit_{SRC_VARIANT}' if SRC_VARIANT else 'bag_zit'   # 산출물 model_name

# 002+ 병렬워커는 결측 2·3단계 fallback을 mean->median으로 monkeypatch한다 (01 노트북=mean).
# 'auto' = source study의 impute_stage23 메타로 자동 판별 / True·False로 강제도 가능.
USE_MEDIAN_IMPUTE_PATCH = 'auto'

# seed pool. 기본값: 30 seed.
SEEDS = list(range(1000, 1030))
N_FOLDS = 5
N_JOBS = 10

# 산출물 위치. 소스 번호별로 분리한다 (seed_sweep_{SRC_NUM}).
RUN_TAG = datetime.now().strftime('run_%m%d_%H%M%S')
OUT_DIR = Path(OUTPUT_DIR) / '01_zit' / 'bag_zit' / f'seed_sweep_{SRC_NUM}' / RUN_TAG
BEST_DIR = OUT_DIR / 'best'
SUMMARY_PATH = OUT_DIR / 'seed_sweep_summary.csv'
RESUME = False

# 저장 정책. fold_models.pkl 하나가 수십 MB라 기본은 best만 저장한다.
SAVE_EVERY_SEED = False
SAVE_BEST = True

# 후처리 집계. ★ BagZIT는 die가 unit_y/n_die 몫을 학습 -> die->unit은 SUM만 정합 (mean/median 무의미).
#   position 'weighted'는 합=1 가중평균이라 sum 스케일과 비호환 -> 후보에서 제외(sum 단일).
BASELINE_AGG = 'sum'
AGG_CANDIDATES = ('sum',)
POSITION_METHOD = 'optuna'        # AGG_CANDIDATES에 'weighted'가 없으므로 실질 no-op (시그니처 호환용)
POSITION_OPTUNA_N_TRIALS = 50

# zero_clip: log-spaced 후보 (0.0001~0.003, 30개).
ZERO_CLIP_RANGE = (0.0001, 0.003)
ZERO_CLIP_N = 30
ZERO_CLIP_LOG_SPACE = True

# isotonic/tail 보정 후보 (05_zit_only 시드스윕과 동일 grid).
ISO_KINDS = ['step', 'pchip']
ISO_WEIGHTS = [0.25, 0.5, 0.75, 1.00, 1.25, 1.50]
TAIL_QS = [0.95, 0.975, 0.99]
TAIL_RESID_QS = [0.75, 0.90]
TAIL_GAINS = [0.0, 0.5, 1.0, 1.5, 2.5]
TAIL_POWERS = [1.0, 2.0]
IQR_TOP_KS = [0, 1, 2]
IQR_MARGIN = 1e-6

# 실행 규모 확인용.
N_SEEDS = len(SEEDS)
N_MODEL_FITS = N_SEEDS * N_FOLDS
N_CALIBRATION_CANDIDATES = 1 + (
    len(ISO_KINDS)
    * len(ISO_WEIGHTS)
    * len(TAIL_QS)
    * len(TAIL_RESID_QS)
    * len(TAIL_GAINS)
    * len(TAIL_POWERS)
    * len(IQR_TOP_KS)
)

OUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_DIR.mkdir(parents=True, exist_ok=True)
print(f'SRC_NUM          = {SRC_NUM}')
print(f'SRC_VARIANT      = {SRC_VARIANT!r}')
print(f'SOURCE_DB_PATH   = {SOURCE_DB_PATH}')
print(f'SOURCE_STUDY     = {SOURCE_STUDY_NAME}')
print(f'OUT_DIR          = {OUT_DIR}')
print(f'MODEL_NAME       = {MODEL_NAME}')
print(f'BASELINE_AGG     = {BASELINE_AGG}  # BagZIT die->unit SUM')
print(f'N_SEEDS          = {N_SEEDS}')
print(f'N_MODEL_FITS     = {N_MODEL_FITS}  # seed {N_SEEDS}개 * {N_FOLDS}-fold')
print(f'N_CAL_CAND/SEED  = {N_CALIBRATION_CANDIDATES}')

SRC_NUM          = 002
SRC_VARIANT      = 'eql'
SOURCE_DB_PATH   = C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\01_zit\bag_zit\hp\002\optuna_jh_bag-zit-eql-final-002.db
SOURCE_STUDY     = bag-zit-eql-final-002
OUT_DIR          = C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\01_zit\bag_zit\seed_sweep_002\run_0601_100619
MODEL_NAME       = bag_zit_eql
BASELINE_AGG     = sum  # BagZIT die->unit SUM
N_SEEDS          = 30
N_MODEL_FITS     = 150  # seed 30개 * 5-fold
N_CAL_CAND/SEED  = 2161


## 2. hp/002 best_params.json 파라미터와 데이터 로드


In [10]:
import ast
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def _parse_user_attr(v):
    """optuna user_attr가 문자열 repr로 박제돼 있으면 원래 타입으로 복원한다.
    003 db는 dict/bool/int가 모두 str로 저장돼 있다. 'zit-only-final-003' 같은
    순수 문자열은 literal_eval이 실패하므로 원본을 그대로 돌려준다."""
    if isinstance(v, str):
        try:
            return ast.literal_eval(v)
        except (ValueError, SyntaxError):
            return v
    return v


# source study의 best trial을 db에서 직접 로드한다 (hp 폴더엔 best_params.json이 없음).
_storage = f'sqlite:///{SOURCE_DB_PATH.as_posix()}'
_study = optuna.load_study(study_name=SOURCE_STUDY_NAME, storage=_storage)
_best = _study.best_trial
_ua = {k: _parse_user_attr(v) for k, v in _study.user_attrs.items()}

# 모델 변종 선택: source가 EQL(Tweedie unit deviance phi)이면 워커의 BagZITEQLRegressor를 로드한다.
# base BagZITboostRegressor(_m_step=Pearson phi)로는 EQL best params를 정확히 재현할 수 없기 때문.
_src_model = str(_ua.get('model') or '')
_src_is_eql = ('eql' in _src_model.lower()) or ('deviance' in _src_model.lower()) or (SRC_VARIANT == 'eql')
if _src_is_eql:
    import importlib.util as _ilu_m
    _eql_wpath = Path(CFG_PROJECT_ROOT) / '3_modeling' / '01_zit' / 'hp' / '02_bag_zit_eql_parallel_hpo.py'
    _espec = _ilu_m.spec_from_file_location('bag_zit_eql_worker', _eql_wpath)
    _emod = _ilu_m.module_from_spec(_espec); _espec.loader.exec_module(_emod)
    ModelClass = _emod.BagZITEQLRegressor
    print(f'[model] source="{_src_model}" -> BagZITEQLRegressor(EQL Tweedie unit deviance) 재현')
else:
    ModelClass = BagZITboostRegressor
    print(f'[model] source="{_src_model}" -> base BagZITboostRegressor 재현')

# best_params 구조는 002와 동일: 모델 HP + tau_pi가 한 dict에 섞여 있다. tau_pi를 분리한다.
_bp = dict(_best.params)
best_tau_pi = float(_bp.pop('tau_pi'))

# clip_y_extreme 키는 003이 소문자('clip_y_extreme'), 002가 대문자('CLIP_Y_EXTREME').
_clip_attr = _ua.get('clip_y_extreme', _ua.get('CLIP_Y_EXTREME', True))
if isinstance(_clip_attr, str):
    _clip_attr = _clip_attr.lower() in {'true', '1', 'yes'}
clip_y_extreme = bool(_clip_attr)

# 이후 셀들이 기대하는 best_params.json 스키마로 source_meta를 합성한다.
# db 직접 로드라 feature_names/n_features/unit_ids_hash 메타는 없어 None (feature check는 자동 skip).
source_meta = {
    'exp_id': _ua.get('exp_id'),
    'model_name': MODEL_NAME,
    'best_trial_number': _best.number,
    'best_oof_rmse': float(_best.value),
    'best_params_resolved': dict(_bp),
    'best_tau_pi': best_tau_pi,
    'effective_pp_params': dict(_ua.get('pp_fixed') or {}),
    'feature_names': None,
    'n_features': None,
    'unit_ids_hash': None,
    'study_meta': {'CLIP_Y_EXTREME': clip_y_extreme, **_ua},
}

base_model_params = dict(source_meta['best_params_resolved'])
pp_fixed = dict(source_meta['effective_pp_params'])

# seed sweep에서 매번 바꿔야 하는 실행 환경 값은 제거한다 (db best_params엔 없지만 방어적으로 유지).
# 남는 값들은 source Optuna winner 하이퍼파라미터로 고정된다.
for k in ['random_state', 'n_jobs', 'verbose', 'device', 'em_tol']:
    base_model_params.pop(k, None)

print('[기준 source study]')
print(f'  exp_id      : {source_meta.get("exp_id")}')
print(f'  best_trial# : {source_meta.get("best_trial_number")}')
print(f'  best_oof    : {source_meta.get("best_oof_rmse"):.9f}')
print(f'  tau_pi      : {best_tau_pi:.9f}')
print(f'  n_features  : {source_meta.get("n_features")}')
print(f'  clip extreme: {clip_y_extreme}')
print(f'  pp_fixed    : {pp_fixed}')

xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

# source 학습 조건과 맞추기 위해 train y의 극단값 clipping도 동일하게 재현한다.
ys_input = {k: v.copy() for k, v in ys.items()}
if clip_y_extreme:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = int((y_raw >= y_raw.max()).sum())
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] train max -> {second_max:.9f}, clipped={n_clipped}')

# 전처리도 source study.user_attrs의 pp_fixed를 그대로 사용한다.
# 002+ 병렬워커는 결측 2·3단계 fallback을 mean->median으로 patch했다(01 노트북=mean).
# USE_MEDIAN_IMPUTE_PATCH='auto'면 source의 impute_stage23 메타로 자동 판별해 동일 적용한다.
_src_uses_median = 'median' in str(_ua.get('impute_stage23') or '').lower()
_apply_median = (USE_MEDIAN_IMPUTE_PATCH is True) or (USE_MEDIAN_IMPUTE_PATCH == 'auto' and _src_uses_median)
if _apply_median:
    try:
        import importlib.util as _ilu, cleaning as _cleaning
        _worker_fname = '02_bag_zit_eql_parallel_hpo.py' if _src_is_eql else '02_bag_zit_parallel_hpo.py'
        _wpath = Path(CFG_PROJECT_ROOT) / '3_modeling' / '01_zit' / 'hp' / _worker_fname
        _spec = _ilu.spec_from_file_location('bag_zit_worker', _wpath)
        _wmod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_wmod)
        _cleaning.impute_spatial = _wmod._impute_spatial_median
        print(f'[impute patch] median fallback 적용 (워커 {_worker_fname}::_impute_spatial_median 로드, source={SOURCE_STUDY_NAME})')
    except Exception as _e:
        print(f'[⚠ impute patch 실패] {_e} -> 표준 mean fallback로 진행 (전처리 불일치 가능)')
elif _src_uses_median:
    print('[⚠ 경고] source는 median fallback인데 USE_MEDIAN_IMPUTE_PATCH가 꺼짐 -> 전처리 불일치 가능')
else:
    print('[impute] 표준 mean fallback — 01 노트북 소스와 일치')

pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=pp_fixed)
xs_train = pp['xs_train']
xs_val = pp['xs_val']
xs_test = pp['xs_test']  # test 예측에도 사용한다.
feat_cols_clean = pp['feat_cols']

# position, die_x, die_y 메타피처까지 추가해야 hp/002·003의 feature set과 맞는다.
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

expected_features = source_meta.get('feature_names')
if expected_features and list(expected_features) != list(feat_cols_clean):
    print('[경고] 현재 재현한 feature_names가 metadata와 다름')
    print(f'  expected={len(expected_features)}, current={len(feat_cols_clean)}')
else:
    print(f'[feature check] feature_names 메타 없음, 재현 feature {len(feat_cols_clean)}개 사용 (pp_fixed 동일 시 source feature set과 일치)')

# 학습 루프에서 반복 접근하므로 numpy float64 행렬로 미리 변환한다.
X_train = xs_train[feat_cols_clean].values.astype(np.float64)
X_val = xs_val[feat_cols_clean].values.astype(np.float64)
X_test = xs_test[feat_cols_clean].values.astype(np.float64)

uid_train_die = xs_train[KEY_COL].values
uid_val_die = xs_val[KEY_COL].values
uid_test_die = xs_test[KEY_COL].values

y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit_s = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit_s = ys_input['test'].set_index(KEY_COL)[TARGET_COL]
y_train_die = xs_train[KEY_COL].map(y_train_unit_s).values.astype(np.float64)

unit_ids_hash = hashlib.sha1(','.join(map(str, ys_input['train'][KEY_COL].unique())).encode()).hexdigest()
print(f'[unit hash] current={unit_ids_hash}')
print(f'[unit hash] source ={source_meta.get("unit_ids_hash")}')
print(f'[data] X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}, units train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}, test={len(y_test_unit_s):,}')

[model] source="BagZIT-EQL parallel HPO worker" -> BagZITEQLRegressor(EQL Tweedie unit deviance) 재현
[기준 source study]
  exp_id      : bag-zit-eql-final-002
  best_trial# : 58
  best_oof    : 0.005494691
  tau_pi      : 0.939766838
  n_features  : None
  clip extreme: True
  pp_fixed    : {'missing_threshold': 0.3, 'corr_threshold': 0.9, 'corr_keep_by': 'std', 'add_indicator': True, 'indicator_threshold': 0.05, 'spatial_max_dist': 6.0, 'post_impute_corr_threshold': 0.96, 'post_impute_corr_keep_by': 'std'}
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729


KeyboardInterrupt: 

## 3. 공통 함수 - 전처리, isotonic tail, 저장


In [ ]:
# 공통 RMSE 계산. 모든 선택 기준은 validation RMSE가 낮은 쪽이다.
def rmse(pred, y):
    pred = np.asarray(pred, dtype=float)
    y = np.asarray(y, dtype=float)
    return float(np.sqrt(np.mean((pred - y) ** 2)))


def clip_nonneg(x):
    return np.clip(np.asarray(x, dtype=float), 0.0, None)


# ZIT의 structural-zero 확률 pi가 tau_pi보다 큰 die는 0으로 강제한다.
def apply_tau_pi(pred_die, pi_die, tau_pi):
    return np.where(pi_die > tau_pi, 0.0, pred_die)


def unit_rmse(unit_df, y_unit_s):
    p = unit_df.set_index(KEY_COL)['pred'].loc[y_unit_s.index].values
    return rmse(p, y_unit_s.values)


def aligned_unit_pred(unit_df, y_unit_s):
    return unit_df.set_index(KEY_COL)['pred'].loc[y_unit_s.index].values.astype(float)


def tune_unit_postprocess_train_val(
    xs_train,
    xs_val,
    xs_test,
    die_pred_train,
    die_pred_val,
    die_pred_test,
    y_train_unit_df,
    y_val_unit_df,
):
    """hp/002 seed sweep과 같은 train/validation 전용 축약판.

    차이: zero_clip 후보 array를 np.arange linear가 아니라 np.logspace로 만든다.
    하한 0.0001부터 상한 0.015까지 log 균등 30개. 작은 양수도 촘촘히 본다.
    """
    y_train_s = y_train_unit_df.set_index(KEY_COL)[TARGET_COL]
    y_val_s = y_val_unit_df.set_index(KEY_COL)[TARGET_COL]
    decisions = {}
    val_history = []

    # 1단계: baseline 집계(mean)에서 출발. validation 개선 시에만 교체.
    train_unit = postprocess.aggregate(xs_train, die_pred_train, BASELINE_AGG)
    val_unit = postprocess.aggregate(xs_val, die_pred_val, BASELINE_AGG)
    test_unit = postprocess.aggregate(xs_test, die_pred_test, BASELINE_AGG)
    cur_val = unit_rmse(val_unit, y_val_s)
    val_history.append((f'baseline_{BASELINE_AGG}', cur_val))

    # 2단계: train OOF에서 best 집계 방식 후보, validation 개선 시에만 채택.
    agg_res = postprocess.find_best_aggregation(
        xs_train,
        die_pred_train,
        y_train_unit_df,
        methods=AGG_CANDIDATES,
        position_method=POSITION_METHOD,
        optuna_n_trials=POSITION_OPTUNA_N_TRIALS,
    )
    best_agg_cand = agg_res['best_method']
    pos_w_cand = agg_res['pos_weights']

    if best_agg_cand == BASELINE_AGG:
        best_agg = BASELINE_AGG
        pos_w = None
        decisions['aggregation'] = f'{BASELINE_AGG} train OOF best -> 유지'
    else:
        cand_train = postprocess.aggregate(xs_train, die_pred_train, best_agg_cand, pos_w_cand)
        cand_val = postprocess.aggregate(xs_val, die_pred_val, best_agg_cand, pos_w_cand)
        cand_test = postprocess.aggregate(xs_test, die_pred_test, best_agg_cand, pos_w_cand)
        cand_val_rmse = unit_rmse(cand_val, y_val_s)
        if cand_val_rmse < cur_val:
            train_unit, val_unit, test_unit = cand_train, cand_val, cand_test
            best_agg, pos_w = best_agg_cand, pos_w_cand
            decisions['aggregation'] = f'{best_agg_cand} 채택 ({cur_val:.9f} -> {cand_val_rmse:.9f})'
            cur_val = cand_val_rmse
        else:
            best_agg, pos_w = BASELINE_AGG, None
            decisions['aggregation'] = f'{best_agg_cand} 거절 ({cur_val:.9f} <= {cand_val_rmse:.9f})'
    val_history.append((f'after_agg({best_agg})', cur_val))

    # 3단계: zero_clip. hp/003 seed sweep의 핵심 변경 - log-spaced 후보.
    zc_arr = np.logspace(
        np.log10(ZERO_CLIP_RANGE[0]),
        np.log10(ZERO_CLIP_RANGE[1]),
        ZERO_CLIP_N,
    )
    zc_res = postprocess.find_best_zero_clip(train_unit, y_train_unit_df, zc_arr, log_space=ZERO_CLIP_LOG_SPACE)
    cand_zc = zc_res['best_threshold']
    cand_train = postprocess.apply_zero_clip(train_unit, cand_zc, log_space=ZERO_CLIP_LOG_SPACE)
    cand_val = postprocess.apply_zero_clip(val_unit, cand_zc, log_space=ZERO_CLIP_LOG_SPACE)
    cand_test = postprocess.apply_zero_clip(test_unit, cand_zc, log_space=ZERO_CLIP_LOG_SPACE)
    cand_val_rmse = unit_rmse(cand_val, y_val_s)

    best_zc = None
    if cand_val_rmse < cur_val:
        train_unit, val_unit, test_unit = cand_train, cand_val, cand_test
        best_zc = cand_zc
        decisions['zero_clip'] = f'{cand_zc:.6f} 채택 ({cur_val:.9f} -> {cand_val_rmse:.9f})'
        cur_val = cand_val_rmse
    else:
        decisions['zero_clip'] = f'{cand_zc:.6f} 거절 ({cur_val:.9f} <= {cand_val_rmse:.9f})'
    val_history.append(('after_zero_clip', cur_val))

    train_rmse = unit_rmse(train_unit, y_train_s)
    return {
        'best_agg': best_agg,
        'pos_weights': pos_w,
        'best_zero_clip': best_zc,
        'zero_clip_log_space': ZERO_CLIP_LOG_SPACE,
        'zero_clip_arr': zc_arr,
        'position_method': POSITION_METHOD,
        'agg_rmses': agg_res['rmse_per_method'],
        'decisions': decisions,
        'val_rmse_history': val_history,
        'train_rmse': train_rmse,
        'val_rmse_final': cur_val,
        'final_train_unit': train_unit,
        'final_val_unit': val_unit,
        'final_test_unit': test_unit,
    }


def iqr_stats(pred, y_true=None):
    pred = np.asarray(pred, dtype=float)
    q1, q3 = np.quantile(pred, [0.25, 0.75])
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    mask = pred > upper
    out = {
        'q1': float(q1),
        'q3': float(q3),
        'iqr': float(iqr),
        'upper_fence': float(upper),
        'n_upper_outliers': int(mask.sum()),
        'max_pred': float(np.max(pred)),
    }
    if y_true is not None and mask.any():
        yy = np.asarray(y_true, dtype=float)[mask]
        out.update({
            'outlier_true_mean': float(np.mean(yy)),
            'outlier_true_max': float(np.max(yy)),
            'outlier_true_ge_q95': int((yy >= np.quantile(y_true, 0.95)).sum()),
        })
    else:
        out.update({'outlier_true_mean': np.nan, 'outlier_true_max': np.nan, 'outlier_true_ge_q95': 0})
    return out


def push_top_k_to_iqr(pred, score, top_k=0, margin=1e-6):
    """예측 rank만 사용하는 batch 변환. y_true는 절대 보지 않는다."""
    pred = np.asarray(pred, dtype=float).copy()
    if top_k <= 0:
        return pred
    q1, q3 = np.quantile(pred, [0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    idx = np.argsort(np.asarray(score, dtype=float))[-int(top_k):]
    pred[idx] = np.maximum(pred[idx], upper + margin)
    return pred


def build_iso_pchip_transform(iso):
    """sklearn IsotonicRegression의 step function을 PCHIP monotonic cubic으로 smoothing한다.

    - knot 값(`X_thresholds_`, `y_thresholds_`)은 그대로 둔다. → 상단 끌어올림 폭은 step과 동일.
    - knot 사이만 PCHIP cubic 보간. → 평탄 plateau가 부드러운 곡선이 된다.
    - PAV가 보장하는 단조 증가성이 PCHIP에서도 유지된다 (PCHIP는 입력 monotonicity를 보존).

    Returns
    -------
    transform : callable. iso.transform과 같은 시그니처(raw -> calibrated).
    """
    x_knots = np.asarray(iso.X_thresholds_, dtype=float)
    y_knots = np.asarray(iso.y_thresholds_, dtype=float)
    # PAV 산출에서 X_thresholds_는 strictly increasing이지만, 방어적 dedupe.
    uniq_mask = np.concatenate([[True], np.diff(x_knots) > 0])
    x_knots = x_knots[uniq_mask]
    y_knots = y_knots[uniq_mask]
    if len(x_knots) < 2:
        # knot이 1개 이하면 PCHIP 불가능 → step 그대로 반환.
        def _fallback(x):
            return iso.transform(np.asarray(x, dtype=float))
        return _fallback, x_knots, y_knots

    pchip = PchipInterpolator(x_knots, y_knots, extrapolate=False)
    lo, hi = float(x_knots[0]), float(x_knots[-1])

    def _transform(x):
        x = np.asarray(x, dtype=float)
        x_clipped = np.clip(x, lo, hi)  # iso의 out_of_bounds='clip'과 같은 동작
        y_out = pchip(x_clipped)
        return np.clip(y_out, 0.0, None)  # y_min=0 강제

    return _transform, x_knots, y_knots


def fit_iso_tail_grid(train_unit, val_unit, test_unit, y_train_s, y_val_s, y_test_s):
    """unit-level 후처리 결과를 raw score로 보고 isotonic/tail 후보를 비교한다.

    hp/002 seed sweep 대비 차이:
    - `iso_kind ∈ {'step', 'pchip'}`이 grid 차원에 추가됨. PCHIP는 step의 knot 값을 그대로 두고 plateau만 smoothing.
    - `ISO_WEIGHTS`에 0.25, 0.5가 추가되어 raw 비중↑ 후보도 함께 탐색.

    train OOF raw -> train y로 fit하고, validation raw에는 transform만 적용한다.
    tail 강화는 raw score 상단부에만 추가 보정을 걸어 RMSE와 outlier 형성을 동시에 노린다.
    """
    raw_train = aligned_unit_pred(train_unit, y_train_s)
    raw_val = aligned_unit_pred(val_unit, y_val_s)
    raw_test = aligned_unit_pred(test_unit, y_test_s)
    y_train = y_train_s.values.astype(float)
    y_val = y_val_s.values.astype(float)
    y_test = y_test_s.values.astype(float)

    rows = []
    best = None

    def add_candidate(name, pred_train, pred_val, pred_test, params, iso_model=None, iso_kind=None, pchip_knots=None):
        nonlocal best
        pred_train = clip_nonneg(pred_train)
        pred_val = clip_nonneg(pred_val)
        pred_test = clip_nonneg(pred_test)
        val_stats = iqr_stats(pred_val, y_val)
        top_idx = int(np.argmax(pred_val))
        rec = {
            'name': name,
            'train_rmse': rmse(pred_train, y_train),
            'val_rmse': rmse(pred_val, y_val),
            # test_rmse는 모니터링용. 후보 선택(best 판정)에는 절대 쓰지 않는다 (val_rmse만 기준).
            'test_rmse': rmse(pred_test, y_test),
            'val_iqr_outliers': val_stats['n_upper_outliers'],
            'val_iqr_upper_fence': val_stats['upper_fence'],
            'val_max_pred': val_stats['max_pred'],
            'val_outlier_true_mean': val_stats['outlier_true_mean'],
            'val_outlier_true_max': val_stats['outlier_true_max'],
            'val_outlier_true_ge_q95': val_stats['outlier_true_ge_q95'],
            'val_top_pred_y_true': float(y_val[top_idx]),
            **params,
        }
        rows.append(rec)
        if best is None or rec['val_rmse'] < best['record']['val_rmse']:
            best = {
                'record': rec,
                'train_pred': pred_train,
                'val_pred': pred_val,
                'test_pred': pred_test,
                'iso_model': iso_model,
                'iso_kind': iso_kind,
                'pchip_knots': pchip_knots,
                'raw_train': raw_train,
                'raw_val': raw_val,
                'raw_test': raw_test,
            }

    # base 후보: iso/tail 없이 postprocess 결과 그대로.
    add_candidate(
        'base_postprocess', raw_train, raw_val, raw_test,
        {'uses_iso': False, 'iso_kind': 'none', 'iso_weight': 0.0,
         'tail_q': np.nan, 'tail_resid_q': np.nan,
         'tail_gain': 0.0, 'tail_power': np.nan, 'iqr_top_k': 0, 'tail_resid_scale': 0.0},
    )

    # PAV fit. step / pchip transform을 둘 다 미리 만들어 두고 itertools.product에서 룩업.
    iso = IsotonicRegression(out_of_bounds='clip', y_min=0)
    iso.fit(raw_train, y_train)
    iso_train_step = iso.transform(raw_train)
    iso_val_step = iso.transform(raw_val)
    iso_test_step = iso.transform(raw_test)
    pchip_transform, pchip_x_knots, pchip_y_knots = build_iso_pchip_transform(iso)
    iso_train_pchip = pchip_transform(raw_train)
    iso_val_pchip = pchip_transform(raw_val)
    iso_test_pchip = pchip_transform(raw_test)

    iso_table = {
        'step':  (iso_train_step,  iso_val_step,  iso_test_step),
        'pchip': (iso_train_pchip, iso_val_pchip, iso_test_pchip),
    }

    for iso_kind, iso_weight, tail_q, tail_resid_q, tail_gain, tail_power, iqr_top_k in itertools.product(
        ISO_KINDS, ISO_WEIGHTS, TAIL_QS, TAIL_RESID_QS, TAIL_GAINS, TAIL_POWERS, IQR_TOP_KS
    ):
        iso_train_arr, iso_val_arr, iso_test_arr = iso_table[iso_kind]

        # iso_weight=1이면 순수 isotonic, 1보다 크면 isotonic 방향으로 더 강하게 당긴다.
        base_train = raw_train + iso_weight * (iso_train_arr - raw_train)
        base_val = raw_val + iso_weight * (iso_val_arr - raw_val)
        base_test = raw_test + iso_weight * (iso_test_arr - raw_test)

        # tail_start 이상 영역만 ramp. tail_resid_scale은 train tail의 양의 residual 분위수.
        tail_start = float(np.quantile(raw_train, tail_q))
        tail_hi = float(np.quantile(raw_train, 0.999))
        tail_denom = max(tail_hi - tail_start, 1e-12)
        tail_mask = raw_train >= tail_start
        if int(tail_mask.sum()) >= 3:
            resid = y_train[tail_mask] - base_train[tail_mask]
            tail_resid_scale = max(0.0, float(np.quantile(resid, tail_resid_q)))
        else:
            tail_resid_scale = 0.0

        def transform(raw, base, ts=tail_start, td=tail_denom, tp=tail_power, tg=tail_gain, trs=tail_resid_scale):
            ramp = np.clip((raw - ts) / td, 0.0, None) ** tp
            return base + tg * trs * ramp

        pred_train = transform(raw_train, base_train)
        pred_val = transform(raw_val, base_val)
        pred_test = transform(raw_test, base_test)

        # IQR outlier push는 y_true를 보지 않고 예측 rank/quantile만 사용 - validation leakage 방지.
        pred_train = push_top_k_to_iqr(pred_train, raw_train, iqr_top_k, IQR_MARGIN)
        pred_val = push_top_k_to_iqr(pred_val, raw_val, iqr_top_k, IQR_MARGIN)
        pred_test = push_top_k_to_iqr(pred_test, raw_test, iqr_top_k, IQR_MARGIN)

        name = f'iso{iso_kind}_w{iso_weight:g}_q{tail_q:g}_rq{tail_resid_q:g}_g{tail_gain:g}_p{tail_power:g}_iqr{iqr_top_k}'
        add_candidate(
            name, pred_train, pred_val, pred_test,
            {
                'uses_iso': True,
                'iso_kind': iso_kind,
                'iso_weight': float(iso_weight),
                'tail_q': float(tail_q),
                'tail_resid_q': float(tail_resid_q),
                'tail_gain': float(tail_gain),
                'tail_power': float(tail_power),
                'iqr_top_k': int(iqr_top_k),
                'tail_start': tail_start,
                'tail_denom': tail_denom,
                'tail_resid_scale': float(tail_resid_scale),
            },
            iso_model=iso,
            iso_kind=iso_kind,
            pchip_knots=(pchip_x_knots, pchip_y_knots) if iso_kind == 'pchip' else None,
        )

    cand = pd.DataFrame(rows).sort_values('val_rmse').reset_index(drop=True)
    iqr12 = cand[cand['val_iqr_outliers'].between(1, 2)].copy()
    best_iqr12 = iqr12.iloc[0].to_dict() if len(iqr12) else None

    best['candidates'] = cand
    best['best_iqr12'] = best_iqr12
    best['raw_train'] = raw_train
    best['raw_val'] = raw_val
    best['raw_test'] = raw_test
    return best

In [ ]:
def json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)


def build_die_df(uid, die_id, position, pi, mu, pred_raw, pred_taupi, y_unit_s):
    out = pd.DataFrame({
        KEY_COL: uid,
        DIE_KEY_COL: die_id,
        'position': position,
        'pi': pi,
        'one_minus_pi': 1.0 - pi,
        'mu': mu,
        'pred_raw': pred_raw,
        'pred_taupi': pred_taupi,
    })
    out[TARGET_COL] = out[KEY_COL].map(y_unit_s)
    return out


def build_unit_output(y_unit_s, pred, pred_base):
    return pd.DataFrame({
        KEY_COL: y_unit_s.index.values,
        'pred': np.asarray(pred, dtype=float),
        'pred_base_postprocess': np.asarray(pred_base, dtype=float),
        TARGET_COL: y_unit_s.values,
    })


def serializable_calibrator(best_cal):
    """best 후보의 파라미터와 isotonic/PCHIP knot을 JSON에 박제한다."""
    rec = dict(best_cal['record'])
    iso = best_cal.get('iso_model')
    if iso is not None:
        rec['iso_x_thresholds'] = iso.X_thresholds_.tolist()
        rec['iso_y_thresholds'] = iso.y_thresholds_.tolist()
    pk = best_cal.get('pchip_knots')
    if pk is not None:
        rec['pchip_x_knots'] = pk[0].tolist()
        rec['pchip_y_knots'] = pk[1].tolist()
    return rec


def save_result_artifacts(res, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    best_cal = res['calibration']
    pp_res = res['postprocess']

    fold_payload = {
        'fold_models': res['fold_models'],
        'feature_names': feat_cols_clean,
        'model_name': MODEL_NAME,
        'n_folds': N_FOLDS,
        'seed': int(res['seed']),
        'fold_model_seeds': res['fold_model_seeds'],
        'em_history_per_fold': res['em_history_per_fold'],
        'source_best_dir': str(SOURCE_BEST_DIR),
        'calibrator': {
            'record': dict(best_cal['record']),
            'iso_model': best_cal.get('iso_model'),
            'iso_kind': best_cal.get('iso_kind'),
            'pchip_knots': best_cal.get('pchip_knots'),
        },
    }
    with open(target_dir / 'fold_models.pkl', 'wb') as f:
        pickle.dump(fold_payload, f)

    raw_train = best_cal['raw_train']
    raw_val = best_cal['raw_val']
    raw_test = best_cal['raw_test']
    build_unit_output(y_train_unit_s, best_cal['train_pred'], raw_train).to_csv(target_dir / 'oof_unit.csv', index=False)
    build_unit_output(y_val_unit_s, best_cal['val_pred'], raw_val).to_csv(target_dir / 'val_unit.csv', index=False)
    build_unit_output(y_test_unit_s, best_cal['test_pred'], raw_test).to_csv(target_dir / 'test_unit.csv', index=False)

    build_die_df(
        uid_train_die,
        xs_train[DIE_KEY_COL].values,
        xs_train['position'].values,
        res['oof_die_pi'],
        res['oof_die_mu'],
        res['oof_die_pred_raw'],
        res['oof_die_pred_taupi'],
        y_train_unit_s,
    ).to_csv(target_dir / 'oof_die.csv', index=False)
    build_die_df(
        uid_val_die,
        xs_val[DIE_KEY_COL].values,
        xs_val['position'].values,
        res['val_die_pi'],
        res['val_die_mu'],
        res['val_die_pred_raw'],
        res['val_die_pred_taupi'],
        y_val_unit_s,
    ).to_csv(target_dir / 'val_die.csv', index=False)
    build_die_df(
        uid_test_die,
        xs_test[DIE_KEY_COL].values,
        xs_test['position'].values,
        res['test_die_pi'],
        res['test_die_mu'],
        res['test_die_pred_raw'],
        res['test_die_pred_taupi'],
        y_test_unit_s,
    ).to_csv(target_dir / 'test_die.csv', index=False)

    best_cal['candidates'].to_csv(target_dir / 'calibration_candidates.csv', index=False)

    meta = {
        'exp_id': f'bag-zit-{SRC_NUM}-seed-sweep-seed{res["seed"]}',
        'model_name': MODEL_NAME,
        'source_best_dir': str(SOURCE_BEST_DIR),
        'source_exp_id': source_meta.get('exp_id'),
        'seed': int(res['seed']),
        'fold_model_seeds': res['fold_model_seeds'],
        'best_params_resolved': res['best_full_params'],
        'best_tau_pi': best_tau_pi,
        'feature_names': feat_cols_clean,
        'n_features': len(feat_cols_clean),
        'n_folds': N_FOLDS,
        'unit_ids_hash': unit_ids_hash,
        'n_units_train': int(len(y_train_unit_s)),
        'n_units_val': int(len(y_val_unit_s)),
        'n_units_test': int(len(y_test_unit_s)),
        'effective_pp_params': pp_fixed,
        'val_rmse': float(res['summary']['val_rmse']),
        'base_val_rmse': float(res['summary']['base_val_rmse']),
        'test_rmse': float(res['summary']['test_rmse']),
        'base_test_rmse': float(res['summary']['base_test_rmse']),
        'postprocess': {
            'best_agg': pp_res['best_agg'],
            'pos_weights': pp_res['pos_weights'].tolist() if pp_res['pos_weights'] is not None else None,
            'best_zero_clip': pp_res['best_zero_clip'],
            'zero_clip_log_space': pp_res['zero_clip_log_space'],
            'zero_clip_arr': pp_res['zero_clip_arr'].tolist(),
            'position_method': pp_res['position_method'],
            'agg_rmses': {k: float(v) for k, v in pp_res['agg_rmses'].items()},
            'train_rmse': float(pp_res['train_rmse']),
            'val_rmse_final': float(pp_res['val_rmse_final']),
            'decisions': pp_res['decisions'],
        },
        'calibration': serializable_calibrator(best_cal),
        'best_iqr12_candidate': best_cal.get('best_iqr12'),
        'created_at': datetime.now().isoformat(timespec='seconds'),
    }
    with open(target_dir / 'best_params.json', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False, default=json_default)

    with open(target_dir / 'summary_record.json', 'w', encoding='utf-8') as f:
        json.dump(res['summary'], f, indent=2, ensure_ascii=False, default=json_default)

    print(f'[저장 완료] {target_dir}')

## 4. seed 1개 재학습 함수


In [ ]:
# seed마다 unit-level KFold split을 새로 만든다. 같은 unit의 4 die가 train/valid에 섞이지 않도록 unit ID 기준.
def make_folds(seed):
    unique_units = y_train_unit_s.index.values
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=int(seed))
    return unique_units, list(kf.split(unique_units))


def params_for_seed(seed, fold_idx):
    p = dict(base_model_params)
    fold_seed = int(seed) * 1009 + int(fold_idx)
    p['random_state'] = fold_seed
    p['n_jobs'] = N_JOBS
    p['verbose'] = -1
    p['device'] = 'cpu'
    p['em_tol'] = 1e-7
    return p, fold_seed


def fit_one_seed(seed):
    seed = int(seed)
    unique_units, folds = make_folds(seed)

    n_train_die = len(X_train)
    n_val_die = len(X_val)

    oof_die_pi = np.full(n_train_die, np.nan)
    oof_die_mu = np.full(n_train_die, np.nan)
    oof_die_pred_raw = np.full(n_train_die, np.nan)

    val_die_pi = np.zeros(n_val_die)
    val_die_mu = np.zeros(n_val_die)
    val_die_pred_raw = np.zeros(n_val_die)

    n_test_die = len(X_test)
    test_die_pi = np.zeros(n_test_die)
    test_die_mu = np.zeros(n_test_die)
    test_die_pred_raw = np.zeros(n_test_die)

    fold_models = []
    fold_model_seeds = []
    em_history_per_fold = []

    t0 = time.time()
    print(f'\n=== seed {seed} ===')
    for fold_idx, (tr_uidx, vl_uidx) in enumerate(folds):
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask = np.isin(uid_train_die, tr_units)
        vl_mask = np.isin(uid_train_die, vl_units)

        params, fold_seed = params_for_seed(seed, fold_idx)
        model = ModelClass(**params)
        model.fit(X_train[tr_mask], y_train_die[tr_mask], unit_id=uid_train_die[tr_mask])

        pi_vl, mu_vl, _ = model.predict_components(X_train[vl_mask])
        pred_vl = clip_nonneg((1.0 - pi_vl) * mu_vl)
        oof_die_pi[vl_mask] = pi_vl
        oof_die_mu[vl_mask] = mu_vl
        oof_die_pred_raw[vl_mask] = pred_vl

        pi_val, mu_val, _ = model.predict_components(X_val)
        val_die_pi += pi_val / N_FOLDS
        val_die_mu += mu_val / N_FOLDS
        val_die_pred_raw += clip_nonneg((1.0 - pi_val) * mu_val) / N_FOLDS

        # test도 같은 5-fold 앙상블 평균. test y는 선택에 절대 쓰지 않고 모니터링/제출용 예측만 만든다.
        pi_test, mu_test, _ = model.predict_components(X_test)
        test_die_pi += pi_test / N_FOLDS
        test_die_mu += mu_test / N_FOLDS
        test_die_pred_raw += clip_nonneg((1.0 - pi_test) * mu_test) / N_FOLDS

        fold_models.append(model)
        fold_model_seeds.append(fold_seed)
        em_history_per_fold.append(getattr(model, 'em_history_', None))
        print(f'  fold {fold_idx + 1}/{N_FOLDS} done, model_seed={fold_seed}, elapsed={time.time() - t0:.0f}s')

    if np.isnan(oof_die_pred_raw).any():
        raise RuntimeError(f'seed {seed}: OOF die prediction has NaN')

    oof_die_pred_taupi = apply_tau_pi(oof_die_pred_raw, oof_die_pi, best_tau_pi)
    val_die_pred_taupi = apply_tau_pi(val_die_pred_raw, val_die_pi, best_tau_pi)
    test_die_pred_taupi = apply_tau_pi(test_die_pred_raw, test_die_pi, best_tau_pi)

    pp_res = tune_unit_postprocess_train_val(
        xs_train=xs_train,
        xs_val=xs_val,
        xs_test=xs_test,
        die_pred_train=oof_die_pred_taupi,
        die_pred_val=val_die_pred_taupi,
        die_pred_test=test_die_pred_taupi,
        y_train_unit_df=ys_input['train'],
        y_val_unit_df=ys_input['validation'],
    )

    cal = fit_iso_tail_grid(pp_res['final_train_unit'], pp_res['final_val_unit'], pp_res['final_test_unit'], y_train_unit_s, y_val_unit_s, y_test_unit_s)
    best_rec = cal['record']
    best_iqr12 = cal.get('best_iqr12')

    # base_test_rmse: postprocess까지만 적용한 test RMSE (calibration 전). base_val_rmse의 짝.
    base_test_rmse = unit_rmse(pp_res['final_test_unit'], y_test_unit_s)

    elapsed = time.time() - t0
    summary = {
        'seed': seed,
        'elapsed_sec': float(elapsed),
        'base_train_rmse': float(pp_res['train_rmse']),
        'base_val_rmse': float(pp_res['val_rmse_final']),
        'base_test_rmse': float(base_test_rmse),
        'val_rmse': float(best_rec['val_rmse']),
        'test_rmse': float(best_rec['test_rmse']),
        'train_rmse': float(best_rec['train_rmse']),
        'calibration_name': best_rec['name'],
        'uses_iso': bool(best_rec['uses_iso']),
        'iso_kind': str(best_rec.get('iso_kind', 'none')),
        'iso_weight': float(best_rec.get('iso_weight', 0.0)),
        'tail_q': float(best_rec.get('tail_q', np.nan)) if not pd.isna(best_rec.get('tail_q', np.nan)) else np.nan,
        'tail_resid_q': float(best_rec.get('tail_resid_q', np.nan)) if not pd.isna(best_rec.get('tail_resid_q', np.nan)) else np.nan,
        'tail_gain': float(best_rec.get('tail_gain', 0.0)),
        'tail_power': float(best_rec.get('tail_power', np.nan)) if not pd.isna(best_rec.get('tail_power', np.nan)) else np.nan,
        'tail_resid_scale': float(best_rec.get('tail_resid_scale', 0.0)),
        'iqr_top_k': int(best_rec.get('iqr_top_k', 0)),
        'val_iqr_outliers': int(best_rec['val_iqr_outliers']),
        'val_iqr_upper_fence': float(best_rec['val_iqr_upper_fence']),
        'val_max_pred': float(best_rec['val_max_pred']),
        'val_outlier_true_mean': float(best_rec['val_outlier_true_mean']) if not pd.isna(best_rec['val_outlier_true_mean']) else np.nan,
        'val_outlier_true_max': float(best_rec['val_outlier_true_max']) if not pd.isna(best_rec['val_outlier_true_max']) else np.nan,
        'val_outlier_true_ge_q95': int(best_rec['val_outlier_true_ge_q95']),
        'val_top_pred_y_true': float(best_rec['val_top_pred_y_true']),
        'best_iqr12_val_rmse': float(best_iqr12['val_rmse']) if best_iqr12 else np.nan,
        'best_iqr12_name': best_iqr12['name'] if best_iqr12 else None,
        'postprocess_best_agg': pp_res['best_agg'],
        'postprocess_best_zero_clip': pp_res['best_zero_clip'],
    }
    print(f'[seed {seed}] base_val={summary["base_val_rmse"]:.9f}, best_val={summary["val_rmse"]:.9f}, '
          f'test={summary["test_rmse"]:.9f}, cal={summary["calibration_name"]}, '
          f'iqr_outliers={summary["val_iqr_outliers"]}, elapsed={elapsed:.0f}s')

    best_full_params, _ = params_for_seed(seed, 0)
    return {
        'seed': seed,
        'summary': summary,
        'postprocess': pp_res,
        'calibration': cal,
        'fold_models': fold_models,
        'fold_model_seeds': fold_model_seeds,
        'em_history_per_fold': em_history_per_fold,
        'best_full_params': best_full_params,
        'oof_die_pi': oof_die_pi,
        'oof_die_mu': oof_die_mu,
        'oof_die_pred_raw': oof_die_pred_raw,
        'oof_die_pred_taupi': oof_die_pred_taupi,
        'val_die_pi': val_die_pi,
        'val_die_mu': val_die_mu,
        'val_die_pred_raw': val_die_pred_raw,
        'val_die_pred_taupi': val_die_pred_taupi,
        'test_die_pi': test_die_pi,
        'test_die_mu': test_die_mu,
        'test_die_pred_raw': test_die_pred_raw,
        'test_die_pred_taupi': test_die_pred_taupi,
    }

## 5. seed sweep 실행


In [ ]:
# RESUME=True이면 기존 summary를 읽고 이미 끝난 seed는 건너뛴다.
if RESUME and SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH)
    rows = summary_df.to_dict('records')
    done_seeds = set(summary_df['seed'].astype(int).tolist())
    best_so_far = float(summary_df['val_rmse'].min()) if len(summary_df) else float('inf')
    print(f'[재개] 기존 {len(summary_df)}개 seed 로드, 현재 best={best_so_far:.9f}')
else:
    rows = []
    done_seeds = set()
    best_so_far = float('inf')


# IQR 상단 이상치 점검은 성능(RMSE)과 분리한다.
#   - 성능: fit_one_seed가 train/val/test 각각 따로 RMSE를 계산한다 (concat 안 함).
#   - 이상치: best calibration 후보 예측을 train+val+test로 concat한 전체 분포에서 IQRx1.5 상단 이상치 유무/개수만 본다.
# iqr_stats와 동일한 fence 정의(q3 + 1.5*IQR, 초과분)를 써서 split별 지표와 기준을 맞춘다.
def concat_iqr_outlier_stats(cal):
    pred = np.concatenate([cal['train_pred'], cal['val_pred'], cal['test_pred']])
    q1, q3 = np.quantile(pred, [0.25, 0.75])
    upper_fence = float(q3 + 1.5 * (q3 - q1))
    n_out = int((pred > upper_fence).sum())
    return {
        'concat_n_total': int(len(pred)),
        'concat_iqr_upper_fence': upper_fence,
        'concat_iqr_outliers': n_out,
        'concat_has_outlier': bool(n_out > 0),
        'concat_max_pred': float(pred.max()),
    }


for seed in SEEDS:
    if int(seed) in done_seeds:
        print(f'[건너뜀] seed {seed}는 이미 summary에 있음')
        continue

    res = fit_one_seed(seed)

    # 성능과 별개로, concat(train+val+test) 분포 기준 IQR 상단 이상치 정보를 summary에 덧붙인다.
    concat_stats = concat_iqr_outlier_stats(res['calibration'])
    res['summary'].update(concat_stats)
    print(f"  [concat IQR] train+val+test n={concat_stats['concat_n_total']}, "
          f"upper_fence={concat_stats['concat_iqr_upper_fence']:.6f}, "
          f"상단 이상치={concat_stats['concat_iqr_outliers']}개, "
          f"max={concat_stats['concat_max_pred']:.6f}")

    rows.append(res['summary'])
    summary_df = pd.DataFrame(rows).sort_values('val_rmse').reset_index(drop=True)
    summary_df.to_csv(SUMMARY_PATH, index=False)

    seed_is_best = res['summary']['val_rmse'] < best_so_far
    if SAVE_EVERY_SEED:
        save_result_artifacts(res, OUT_DIR / 'seeds' / f'seed_{int(seed)}')
    if SAVE_BEST and seed_is_best:
        best_so_far = float(res['summary']['val_rmse'])
        save_result_artifacts(res, BEST_DIR)
        print(f'[새 best] seed={seed}, val_rmse={best_so_far:.9f}')

    del res
    gc.collect()

print('\n[완료]')
display(pd.read_csv(SUMMARY_PATH).sort_values('val_rmse').head(20))
print(f'best 산출물: {BEST_DIR}')


=== seed 1000 ===


KeyboardInterrupt: 

## 6. 결과 확인


In [ ]:
summary = pd.read_csv(SUMMARY_PATH).sort_values('val_rmse').reset_index(drop=True)
display(summary.head(30))

# isotonic kind별 best 분포 확인 - step vs pchip 중 어느 쪽이 더 자주 best가 됐는지.
if 'iso_kind' in summary.columns:
    print('\n[iso_kind 분포]')
    display(summary['iso_kind'].value_counts())
    print('\n[iso_kind별 평균 val_rmse]')
    display(summary.groupby('iso_kind')['val_rmse'].agg(['count', 'mean', 'min']))

# val_rmse는 다수 후보 중 best를 val로 골라서 낙관적(selection bias). test_rmse가 정직한 일반화 지표.
if 'test_rmse' in summary.columns:
    print('\n[val vs test 갭 - 클수록 val 과적합 신호]')
    gap = summary['test_rmse'] - summary['val_rmse']
    print(f'  mean val_rmse  = {summary["val_rmse"].mean():.9f}')
    print(f'  mean test_rmse = {summary["test_rmse"].mean():.9f}')
    print(f'  mean gap(test-val) = {gap.mean():.9f}')

# IQR 상단 이상치는 성능과 분리해 train+val+test를 concat한 전체 분포 기준으로 본다.
if 'concat_iqr_outliers' in summary.columns:
    print()
    print('[concat(train+val+test) IQRx1.5 상단 이상치]')
    n_seed = len(summary)
    n_have = int((summary['concat_iqr_outliers'] > 0).sum())
    print(f'  이상치 보유 seed = {n_have}/{n_seed}')
    print(f'  seed별 이상치 개수: min={int(summary["concat_iqr_outliers"].min())}, '
          f'median={summary["concat_iqr_outliers"].median():.1f}, '
          f'max={int(summary["concat_iqr_outliers"].max())}')
    _ccols = [c for c in ['seed', 'val_rmse', 'test_rmse', 'concat_iqr_outliers',
                          'concat_iqr_upper_fence', 'concat_max_pred', 'concat_n_total']
              if c in summary.columns]
    display(summary[_ccols].sort_values('val_rmse').head(30))

iqr12 = summary[summary['val_iqr_outliers'].between(1, 2)].copy()
print('\n[validation RMSE 기준 best]')
display(summary.head(1))

print('\n[IQR upper outlier 1~2개 조건 best]')
if len(iqr12):
    display(iqr12.sort_values('val_rmse').head(10))
else:
    print('아직 IQR upper outlier 1~2개 조건을 만족하는 후보가 없습니다.')

print(f'OUT_DIR  = {OUT_DIR}')
print(f'BEST_DIR = {BEST_DIR}')